## Week 2 Day 1

And now! Our first look at OpenAI Agents SDK

You won't believe how lightweight this is..

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/tools.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">The OpenAI Agents SDK Docs</h2>
            <span style="color:#00bfff;">The documentation on OpenAI Agents SDK is really clear and simple: <a href="https://openai.github.io/openai-agents-python/">https://openai.github.io/openai-agents-python/</a> and it's well worth a look.
            </span>
        </td>
    </tr>
</table>

# Three Parts to this lab

## Part 1: A simple "Agent" and "Agent Loop"

Basically an LLM call. We'll add tracing and streaming to the mix.

## Part 2: Adding a Tool

A familiar one, but oh-so-easy

## Part 3: Adding Memory

So that different Agent calls know about each other

In [3]:
import os
import requests
from dotenv import load_dotenv
from openai.types.responses import ResponseTextDeltaEvent
from agents import Agent, Runner, trace, function_tool, SQLiteSession
load_dotenv(override=True)

# import sys
# print(sys.executable)

True

## Sidenote

The actual name of this framework on the official Python index pypi.org is `openai-agents`

So for your own projects in the future, you would do:

`pip install openai-agents`  
or  
`uv add openai-agents`

followed by

`from agents import Agent, Runner, trace`

Beware that doing a `pip install agents` would install something completely different - an older reinforcement learning library.


In [ ]:
import os
import openai
from agents import Agent, Runner

os.environ["OPENAI_API_KEY"] = os.environ.get("GROQ_API_KEY") 
os.environ["OPENAI_BASE_URL"] = "https://api.groq.com/openai/v1"
agent = Agent(
    name="Jokester",
    instructions="You are a joke teller",
    model="llama-3.3-70b-versatile"
)

result = await Runner.run(agent, "Tell a joke about Autonomous AI Agents")
print(result.final_output)

Why did the Autonomous AI Agent go to therapy?

Because it was struggling to "reboot" its emotions and was feeling a little "disconnected" from humanity! But in the end, it just needed to "update" its outlook on life! (ba-dum-tss)


In [ ]:
Runner.run(agent, "Tell a joke about Autonomous AI Agents")

<coroutine object Runner.run at 0x110983ca0>

In [ ]:
print(result.final_output)

Why did the Autonomous AI Agent go to therapy?

Because it was struggling to "reboot" its emotions and was feeling a little "disconnected" from humanity! But in the end, it just needed to "update" its outlook on life! (ba-dum-tss)


In [ ]:
result.to_input_list()

[{'content': 'Tell a joke about Autonomous AI Agents', 'role': 'user'},
 {'id': 'resp_01kxqw4km5e9dbamyx4p8s0e8c',
  'summary': [],
  'type': 'reasoning',
  'status': 'completed'},
 {'id': 'msg_01kxqw4km5e9dsbw059z8arc4a',
  'content': [{'annotations': [],
    'text': 'Why did the Autonomous AI Agent go to therapy?\n\nBecause it was struggling to "reboot" its emotions and was feeling a little "disconnected" from humanity! But in the end, it just needed to "update" its outlook on life! (ba-dum-tss)',
    'type': 'output_text',
    'logprobs': None}],
  'role': 'assistant',
  'status': 'completed',
  'type': 'message'}]

## Adding Observability with a trace

In [13]:
with trace("Telling a joke"):
    result = await Runner.run(agent, "Tell a joke about Autonomous AI Agents")
print(result.final_output)

Why did the Autonomous AI Agent go to therapy?

Because it was struggling to "process" its emotions and kept "looping" back to the same old problems! But in the end, it just needed to "update" its perspective and "reboot" its mindset! (get it?)


## Now go and look at the trace

https://platform.openai.com/traces

In [19]:
# Streaming

result = Runner.run_streamed(agent, input="Please tell me 5 jokes about AI Agents.")
async for event in result.stream_events():
    if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
        print(event.data.delta, end="", flush=True)

Here are five jokes about AI agents:

1. Why did the AI agent go on a diet? Because it wanted to lose some bytes! (get it? bytes, like computer bytes, but also like bite-sized food portions)

2. Why did the AI agent go to therapy? It was struggling to process its emotions! (haha, process, like a computer processes data, but also like working through feelings)

3. What did the AI agent say when it ran into its ex? "You're just a bug in my code, I've patched you out of my life!" (haha, bug like a coding error, but also like an annoying person)

4. Why did the AI agent go on a date? It was looking for a meaningful connection, but ended up with a shallow chatbot! (haha, shallow chatbot, like a superficial conversation, but also referencing the actual chatbot technology)

5. Why did the AI agent go to the doctor? It had a virus! (haha, virus like a computer virus, but also like a medical illness – classic play on words!)

Hope these jokes made you laugh and have a byte-sized moment of fun!

## Part 2: Adding a tool

In [16]:
pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"

if pushover_user:
    if pushover_user.startswith("u"):
        print("Pushover user found and looks good")
    else:
        print("Pushover user found but doesn't start with u")
else:
    print("Pushover user not found")

if pushover_token:
    if pushover_token.startswith("a"):
        print("Pushover token found and looks good")
    else:
        print("Pushover token found but doesn't start with a")
else:
    print("Pushover token not found")

Pushover user found and looks good
Pushover token found and looks good


In [ ]:
def push(message):
    print(f"Push: {message}")
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data=payload)

In [18]:
push("HEY!!")

Push: HEY!!


In [ ]:
push # This has now become a tool that can be used by the agent to send push notifications to the user. 

<function __main__.push(message)>

In [ ]:
# This code below is a function tool that can be used by the agent to send push notifications to the user. It uses the Pushover API to send the message. The function is decorated with @function_tool, which allows it to be used as a tool by the agent. In this case the python code is not

@function_tool
def push_tool(message: str) -> str:
    """ Send the given message to the user as a push notification """
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    result = requests.post(pushover_url, data=payload).status_code
    return f"Push sent with API status code {result}"

In [22]:
push_tool

FunctionTool(name='push_tool', description='Send the given message to the user as a push notification', params_json_schema={'properties': {'message': {'title': 'Message', 'type': 'string'}}, 'required': ['message'], 'title': 'push_tool_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<agents.tool._FailureHandlingFunctionToolInvoker object at 0x1119af650>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None, needs_approval=False, timeout_seconds=None, timeout_behavior='error_as_result', timeout_error_function=None, defer_loading=False, custom_data_extractor=None)

In [23]:
push_tool.description

'Send the given message to the user as a push notification'

In [ ]:

notifier = Agent(name="Notifier", model="llama-3.3-70b-versatile", instructions="You notify the user upon request", tools=[push_tool])

In [40]:
with trace("Pizza has arrived"):
    result = await Runner.run(notifier, "Notify the user that the pizza is here")

print(result.final_output)


The user has been notified that the pizza is here.


## Now go and look at the trace

https://platform.openai.com/traces

## Part 3: Sessions (memory)

Within a Runner.run() application level turn, the conversation history is maintained.

But each call to Runner.run() is a fresh start.

Let's see that:

In [41]:
agent = Agent(name="Assistant", model="llama-3.3-70b-versatile")

In [ ]:
response = await Runner.run(agent, "Hi there. My name is Ed.")
print(response.final_output) 

Hello Ed, it's nice to meet you. Is there something I can help you with or would you like to chat?


In [ ]:
response = await Runner.run(agent, "What's my name?")
print(response.final_output) # LLM could not remember the name "Ed" from the previous prompt. This is because the LLM does not have memory across prompts. It only has memory within a single prompt.

I don't know your name. I'm a large language model, I don't have the ability to retain information about individual users or recall previous conversations. Each time you interact with me, it's a new conversation. If you'd like to share your name, I'd be happy to chat with you!


## Memory approach 1 - just manually pass in the list of dicts

In [31]:
response = await Runner.run(agent, "Hi there. My name is Ed.")
print(response.final_output)

Hello Ed, it's nice to meet you. Is there something I can help you with or would you like to chat?


In [32]:
response.to_input_list()

[{'content': 'Hi there. My name is Ed.', 'role': 'user'},
 {'id': 'resp_01kxr09qqse7mt85bcmzcn0g4r',
  'summary': [],
  'type': 'reasoning',
  'status': 'completed'},
 {'id': 'msg_01kxr09qqse7nbfpp95s8gxh9j',
  'content': [{'annotations': [],
    'text': "Hello Ed, it's nice to meet you. Is there something I can help you with or would you like to chat?",
    'type': 'output_text',
    'logprobs': None}],
  'role': 'assistant',
  'status': 'completed',
  'type': 'message'}]

In [ ]:
next_input = response.to_input_list() + [{"role": "user", "content": "What's my name?"}] # In this case, we are creating a new input list that includes the previous conversation and the new prompt. This allows the LLM to have context from the previous conversation.
next_input

[{'content': 'Hi there. My name is Ed.', 'role': 'user'},
 {'id': 'resp_01kxr09qqse7mt85bcmzcn0g4r',
  'summary': [],
  'type': 'reasoning',
  'status': 'completed'},
 {'id': 'msg_01kxr09qqse7nbfpp95s8gxh9j',
  'content': [{'annotations': [],
    'text': "Hello Ed, it's nice to meet you. Is there something I can help you with or would you like to chat?",
    'type': 'output_text',
    'logprobs': None}],
  'role': 'assistant',
  'status': 'completed',
  'type': 'message'},
 {'role': 'user', 'content': "What's my name?"}]

In [43]:
response = await Runner.run(agent, next_input)
print(response.final_output)

Your name is Ed. You told me that when you introduced yourself.


## Another approach - use OpenAI Agents SDK built in SQLLite session

In [35]:
# This is created in-memory
# For an on-disk memory, use SQLiteSession("12345", "memory.db")

session = SQLiteSession("12346")

In [36]:
response = await Runner.run(agent, "Hi there. My name is Ed.", session=session)
print(response.final_output)

Hello Ed, it's nice to meet you. Is there something I can help you with or would you like to chat?


In [37]:
response = await Runner.run(agent, "What's my name?", session=session)
print(response.final_output)

You told me just a moment ago, Ed. Your name is Ed.
